# 04 - XGBoost tabular baseline nhe

Notebook nay train baseline XGBoost tren feature tabular nho tu output notebook 01.

Muc tieu:

- Khong grid search
- Khong feature qua rong nhu one-hot `GeneSymbol`
- Mac dinh chay CPU `tree_method="hist"` de an toan voi may 6GB VRAM
- Co `RUN_FRACTION` de test nhanh neu can
- Co feature importance va SHAP sample nho de giai thich

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path("/mnt/MyData/Bioinformatics/Project")
PROCESSED_DIR = PROJECT_DIR / "processed"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

TABULAR_PATH = PROCESSED_DIR / "clinvar_tabular_cnn_aligned.parquet"

ANNOTATION_CANDIDATES = [
    PROCESSED_DIR / "clinvar_external_annotations.parquet",
    PROJECT_DIR / "Data" / "annotations" / "variant_annotations.parquet",
    PROJECT_DIR / "Data" / "annotations" / "variant_annotations.csv",
]

if not TABULAR_PATH.exists():
    raise FileNotFoundError(f"Chua co {TABULAR_PATH}. Hay chay notebook 07 hoac tao aligned parquet truoc.")

TABULAR_PATH, ANNOTATION_CANDIDATES

## 1. Cau hinh

Mac dinh dung CPU. Neu sau nay muon thu GPU, dat `USE_GPU = True`; voi 6GB VRAM, CPU van la lua chon an toan hon.

In [ ]:
RANDOM_STATE = 42
RUN_FRACTION = 1.0  # 1.0 = full 460k rows; 0.1 = smoke test nhanh
USE_GPU = False

# Cau hinh nhe, khong grid search.
XGB_PARAMS = {
    "n_estimators": 400,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "tree_method": "hist",
}

if USE_GPU:
    XGB_PARAMS["device"] = "cuda"

XGB_PARAMS

## 2. Load metadata va tao feature nho

Feature baseline nho van giu cac cot san co. Notebook nay co them ho tro 3 nhom annotation dau tien neu co file ngoai:

- `consequence`: hau qua variant tu VEP/SnpEff/ANNOVAR, vi du `missense_variant`, `synonymous_variant`, `stop_gained`, `splice_donor_variant`.
- `gnomAD_AF`: allele frequency trong gnomAD.
- `CADD_PHRED` va `SpliceAI_max`: diem du doan tac hai tong quat va splicing.

Notebook se tim annotation theo thu tu:

1. `processed/clinvar_external_annotations.parquet`
2. `Data/annotations/variant_annotations.parquet`
3. `Data/annotations/variant_annotations.csv`

Schema can co khoa join:

`Chromosome`, `PositionVCF`, `ReferenceAlleleVCF`, `AlternateAlleleVCF`

Va cac cot annotation tuy chon:

`consequence`, `gnomAD_AF`, `CADD_PHRED`, `SpliceAI_max`

Neu chua co file annotation, cac cot nay se duoc tao voi gia tri `unknown`/missing flag de notebook van chay duoc.

In [ ]:
df = pd.read_parquet(TABULAR_PATH)
assert "Y" in df.columns

y = df["Y"].to_numpy(dtype=np.int8)
df = df.copy()

variant_key_cols = [
    "Chromosome",
    "PositionVCF",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF",
]

annotation_path = next((path for path in ANNOTATION_CANDIDATES if path.exists()), None)
if annotation_path is None:
    print("Chua tim thay external annotation file. Se dung annotation placeholder: consequence=unknown, score missing.")
    annotation_df = None
else:
    print(f"Loading annotation file: {annotation_path}")
    if annotation_path.suffix == ".parquet":
        annotation_df = pd.read_parquet(annotation_path)
    else:
        annotation_df = pd.read_csv(annotation_path)

    missing_key_cols = [col for col in variant_key_cols if col not in annotation_df.columns]
    if missing_key_cols:
        raise ValueError(f"Annotation file thieu key columns: {missing_key_cols}")

    available_annotation_cols = [
        col for col in ["consequence", "gnomAD_AF", "CADD_PHRED", "SpliceAI_max"]
        if col in annotation_df.columns
    ]
    if not available_annotation_cols:
        raise ValueError("Annotation file khong co cot annotation nao trong: consequence, gnomAD_AF, CADD_PHRED, SpliceAI_max")

    annotation_df = annotation_df[variant_key_cols + available_annotation_cols].copy()
    for col in variant_key_cols:
        annotation_df[col] = annotation_df[col].astype(str)
    annotation_df = annotation_df.drop_duplicates(subset=variant_key_cols, keep="first")

    for col in variant_key_cols:
        df[col] = df[col].astype(str)
    df = df.merge(annotation_df, on=variant_key_cols, how="left")

if "consequence" not in df.columns:
    df["consequence"] = "unknown"
else:
    df["consequence"] = df["consequence"].fillna("unknown").astype(str)

for col in ["gnomAD_AF", "CADD_PHRED", "SpliceAI_max"]:
    if col not in df.columns:
        df[col] = np.nan
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.shape)
print("Annotation coverage:")
print({
    "consequence_known": int(df["consequence"].ne("unknown").sum()),
    "gnomAD_AF_non_null": int(df["gnomAD_AF"].notna().sum()),
    "CADD_PHRED_non_null": int(df["CADD_PHRED"].notna().sum()),
    "SpliceAI_max_non_null": int(df["SpliceAI_max"].notna().sum()),
})
df.head()

In [ ]:
feature_df = pd.DataFrame(index=df.index)
feature_df["Chromosome"] = df["Chromosome"].astype(str)
feature_df["ReferenceAlleleVCF"] = df["ReferenceAlleleVCF"].astype(str)
feature_df["AlternateAlleleVCF"] = df["AlternateAlleleVCF"].astype(str)
feature_df["ref_alt"] = df["ref_alt"].astype(str)
feature_df["ReviewStatusSimple"] = df["ReviewStatusSimple"].astype(str)
feature_df["OriginSimple"] = df["OriginSimple"].fillna("unknown").astype(str)
feature_df["consequence"] = df["consequence"].fillna("unknown").astype(str)

numeric_source_cols = [
    "PositionVCF_int",
    "NumberSubmitters_int",
    "SubmitterCategories_int",
    "is_transition",
    "is_transversion",
    "variant_span",
    "has_rs",
    "tested_in_gtr_flag",
    "LastEvaluated_year",
    "phenotype_count",
    "rcv_count",
    "scv_count",
    "has_omim_other_id",
    "chromosome_is_autosome",
    "chromosome_is_sex",
    "chromosome_is_mt",
]
for col in numeric_source_cols:
    feature_df[col] = pd.to_numeric(df[col], errors="coerce").fillna(-1).astype("float32")

for source_col in ["gnomAD_AF", "CADD_PHRED", "SpliceAI_max"]:
    missing_col = f"{source_col}_missing"
    filled_col = f"{source_col}_filled"
    feature_df[missing_col] = df[source_col].isna().astype("int8")
    feature_df[filled_col] = df[source_col].fillna(-1).astype("float32")

categorical_cols = [
    "Chromosome",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF",
    "ref_alt",
    "ReviewStatusSimple",
    "OriginSimple",
    "consequence",
]
numeric_cols = [col for col in feature_df.columns if col not in categorical_cols]

print(feature_df.shape)
display(feature_df.head())
display(feature_df[categorical_cols].nunique())

## 3. Encode categorical feature

Dung `OneHotEncoder` cho feature categorical nho. Khong one-hot `GeneSymbol` trong baseline nay.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report, confusion_matrix, precision_recall_curve
from xgboost import XGBClassifier

indices = np.arange(len(y))
train_idx, temp_idx = train_test_split(indices, test_size=0.30, random_state=RANDOM_STATE, stratify=y)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=RANDOM_STATE, stratify=y[temp_idx])

if RUN_FRACTION < 1.0:
    train_idx, _ = train_test_split(
        train_idx,
        train_size=RUN_FRACTION,
        random_state=RANDOM_STATE,
        stratify=y[train_idx],
    )

X_train = feature_df.iloc[train_idx]
y_train = y[train_idx]
X_val = feature_df.iloc[val_idx]
y_val = y[val_idx]
X_test = feature_df.iloc[test_idx]
y_test = y[test_idx]

print(f"train: {X_train.shape}, labels={dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"val:   {X_val.shape}, labels={dict(zip(*np.unique(y_val, return_counts=True)))}")
print(f"test:  {X_test.shape}, labels={dict(zip(*np.unique(y_test, return_counts=True)))}")

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

xgb_model = XGBClassifier(**XGB_PARAMS)

model = Pipeline([
    ("preprocess", preprocessor),
    ("xgb", xgb_model),
])

model

## 4. Train XGBoost baseline

Khong grid search. Train mot cau hinh mac dinh nhe.

In [ ]:
import time

start_time = time.time()
model.fit(X_train, y_train)
train_seconds = time.time() - start_time

print(f"Train time: {train_seconds / 60:.2f} minutes")

## 5. Evaluate

In [ ]:
proba_val = model.predict_proba(X_val)[:, 1]
proba_test = model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(np.int8)

metrics = {
    "val_pr_auc": average_precision_score(y_val, proba_val),
    "val_roc_auc": roc_auc_score(y_val, proba_val),
    "test_pr_auc": average_precision_score(y_test, proba_test),
    "test_roc_auc": roc_auc_score(y_test, proba_test),
    "test_accuracy_05": (pred_test == y_test).mean(),
}
metrics

In [ ]:
print(classification_report(y_test, pred_test, target_names=["benign", "pathogenic"]))

confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test, pred_test),
    index=["true_benign", "true_pathogenic"],
    columns=["pred_benign", "pred_pathogenic"],
)
confusion_matrix_df

## 6. Threshold analysis

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, proba_test)
pr_table = pd.DataFrame({
    "threshold": np.r_[thresholds, 1.0],
    "precision": precision,
    "recall": recall,
})

recall_at_precision_50 = pr_table.loc[pr_table["precision"].ge(0.50), "recall"].max()
precision_at_recall_80 = pr_table.loc[pr_table["recall"].ge(0.80), "precision"].max()

f1 = 2 * pr_table["precision"] * pr_table["recall"] / (pr_table["precision"] + pr_table["recall"] + 1e-12)
f2 = 5 * pr_table["precision"] * pr_table["recall"] / (4 * pr_table["precision"] + pr_table["recall"] + 1e-12)

best_f1_row = pr_table.iloc[f1.idxmax()].copy()
best_f1_row["f1"] = f1.max()
best_f2_row = pr_table.iloc[f2.idxmax()].copy()
best_f2_row["f2"] = f2.max()

print(f"Recall at precision >= 0.50: {recall_at_precision_50:.4f}")
print(f"Precision at recall >= 0.80: {precision_at_recall_80:.4f}")
print("Best F1 threshold:")
display(best_f1_row.to_frame().T)
print("Best F2 threshold:")
display(best_f2_row.to_frame().T)

## 7. Feature importance

Feature importance o day la XGBoost gain. Day la giai thich global, khong phai per-variant.

In [ ]:
encoded_feature_names = model.named_steps["preprocess"].get_feature_names_out()
booster = model.named_steps["xgb"].get_booster()
importance_gain = booster.get_score(importance_type="gain")

importance_df = pd.DataFrame({
    "feature": encoded_feature_names,
    "xgb_feature": [f"f{i}" for i in range(len(encoded_feature_names))],
})
importance_df["gain"] = importance_df["xgb_feature"].map(importance_gain).fillna(0.0)
importance_df = importance_df.sort_values("gain", ascending=False)

importance_df.head(30)

## 8. SHAP sample nho

SHAP co the ton RAM/thoi gian. Mac dinh chi tinh tren 1000 mau test.

In [ ]:
import shap

SHAP_SAMPLE_SIZE = 1000
sample_size = min(SHAP_SAMPLE_SIZE, len(X_test))
shap_sample = X_test.sample(sample_size, random_state=RANDOM_STATE)
shap_sample_encoded = model.named_steps["preprocess"].transform(shap_sample)

explainer = shap.TreeExplainer(model.named_steps["xgb"])
shap_values = explainer.shap_values(shap_sample_encoded)

print(f"SHAP sample encoded shape: {shap_sample_encoded.shape}")
print(f"SHAP values shape: {np.asarray(shap_values).shape}")

In [ ]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance_df = pd.DataFrame({
    "feature": encoded_feature_names,
    "mean_abs_shap": mean_abs_shap,
}).sort_values("mean_abs_shap", ascending=False)

shap_importance_df.head(30)

## 9. Save outputs

In [ ]:
import joblib

model_path = MODEL_DIR / "clinvar_xgboost_tabular_baseline.joblib"
prediction_path = PROCESSED_DIR / "xgboost_tabular_baseline_test_predictions.parquet"
importance_path = PROCESSED_DIR / "xgboost_tabular_baseline_feature_importance.parquet"
shap_importance_path = PROCESSED_DIR / "xgboost_tabular_baseline_shap_importance.parquet"
threshold_path = PROCESSED_DIR / "xgboost_tabular_baseline_threshold_table.parquet"

prediction_df = df.iloc[test_idx][[
    "GeneSymbol", "ClinicalSignificance", "Y", "Chromosome", "PositionVCF",
    "ReferenceAlleleVCF", "AlternateAlleleVCF", "ReviewStatus"
]].copy()
prediction_df["pred_proba_pathogenic"] = proba_test
prediction_df["pred_label_05"] = pred_test

joblib.dump(model, model_path)
prediction_df.to_parquet(prediction_path, index=False, engine="pyarrow")
importance_df.to_parquet(importance_path, index=False, engine="pyarrow")
shap_importance_df.to_parquet(shap_importance_path, index=False, engine="pyarrow")
pr_table.to_parquet(threshold_path, index=False, engine="pyarrow")

print(f"Saved model: {model_path}")
print(f"Saved predictions: {prediction_path}")
print(f"Saved feature importance: {importance_path}")
print(f"Saved SHAP importance: {shap_importance_path}")
print(f"Saved threshold table: {threshold_path}")